# 📡 Seeing Signals — Part II
## CNN Baseline: Multi-Task Constellation Classification

**Workflow:**
1. Configure dataset and output directories
2. Dataset class & splits (70/15/15)
3. Custom CNN architecture (multi-task)
4. Training loop
5. Evaluation: accuracy, confusion matrices
6. Accuracy vs SNR analysis
7. Generalization test

In [ ]:
from pathlib import Path

DATASET_DIR = Path('../data/generated')
OUTPUT_DIR = Path('../results/cnn')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset directory: {DATASET_DIR.resolve()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')


## Cell 2 — Install & Imports

In [ ]:
%matplotlib inline
!pip install torch torchvision tqdm pandas scikit-learn matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import json

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 3 — Label Encoding

In [ ]:
# Load labels
df = pd.read_csv(DATASET_DIR / 'labels.csv')
print(f'Total samples: {len(df)}')

# Label encoders
MOD_CLASSES = sorted(df['modulation'].unique().tolist())
SNR_CLASSES = ['low', 'medium', 'high']
PN_CLASSES  = ['none', 'mild', 'moderate', 'severe']
IQ_CLASSES  = ['none', 'mild', 'moderate', 'severe']
JAM_CLASSES = ['none', 'present']

MOD2IDX = {m: i for i, m in enumerate(MOD_CLASSES)}
SNR2IDX = {s: i for i, s in enumerate(SNR_CLASSES)}
PN2IDX  = {p: i for i, p in enumerate(PN_CLASSES)}
IQ2IDX  = {q: i for i, q in enumerate(IQ_CLASSES)}
JAM2IDX = {j: i for i, j in enumerate(JAM_CLASSES)}

print(f'Modulation classes ({len(MOD_CLASSES)}): {MOD_CLASSES}')
print(f'SNR classes: {SNR_CLASSES}')
print(f'Phase noise classes: {PN_CLASSES}')
print(f'IQ imbalance classes: {IQ_CLASSES}')
print(f'Jamming classes: {JAM_CLASSES}')

## Cell 4 — Dataset Class & Splits

In [ ]:
class ConstellationDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = {
            'modulation': MOD2IDX[row['modulation']],
            'snr_range':  SNR2IDX[row['snr_range']],
            'phase_noise': PN2IDX[row['phase_noise']],
            'iq_imbalance': IQ2IDX[row['iq_imbalance']],
            'jamming':     JAM2IDX[row['jamming']],
            'snr_db':      float(row['snr_db']),
        }
        return img, labels

# Train/Val/Test split: 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42,
                                      stratify=df['modulation'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42,
                                      stratify=temp_df['modulation'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

IMG_DIR = DATASET_DIR / 'images'
train_ds = ConstellationDataset(train_df, IMG_DIR, train_transform)
val_ds   = ConstellationDataset(val_df,   IMG_DIR, val_transform)
test_ds  = ConstellationDataset(test_df,  IMG_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print('DataLoaders OK')

## Cell 5 — Custom CNN Architecture (Multi-Task)

In [ ]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ConstellationCNN(nn.Module):
    """
    Custom multi-task CNN for constellation classification.
    Input:  RGB image 224×224
    Output: 5 classification heads
        - modulation   : 16 classes
        - snr_range    :  3 classes
        - phase_noise  :  4 classes
        - iq_imbalance :  4 classes
        - jamming      :  2 classes
    """
    def __init__(self):
        super().__init__()

        # Shared feature extractor
        self.features = nn.Sequential(
            ConvBlock(3,   32),   # 224→112
            ConvBlock(32,  64),   # 112→56
            ConvBlock(64, 128),   # 56→28
            ConvBlock(128, 256),  # 28→14
            ConvBlock(256, 256),  # 14→7
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # 7→1
        self.dropout = nn.Dropout(0.5)

        # Shared FC
        self.shared_fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        # Task-specific heads
        self.head_mod = nn.Linear(512, 16)   # modulation
        self.head_snr = nn.Linear(512,  3)   # SNR range
        self.head_pn  = nn.Linear(512,  4)   # phase noise
        self.head_iq  = nn.Linear(512,  4)   # IQ imbalance
        self.head_jam = nn.Linear(512,  2)   # jamming

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        x = self.shared_fc(x)
        return {
            'modulation':   self.head_mod(x),
            'snr_range':    self.head_snr(x),
            'phase_noise':  self.head_pn(x),
            'iq_imbalance': self.head_iq(x),
            'jamming':      self.head_jam(x),
        }


model = ConstellationCNN().to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(model)

## Cell 6 — Training Loop

In [ ]:
# Loss weights (modulation is the primary task)
TASK_WEIGHTS = {
    'modulation':   2.0,
    'snr_range':    1.0,
    'phase_noise':  1.0,
    'iq_imbalance': 1.0,
    'jamming':      1.0,
}
TASKS = list(TASK_WEIGHTS.keys())

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

N_EPOCHS = 20
history  = {'train_loss': [], 'val_loss': [], 'val_acc_mod': []}

best_val_acc = 0.0
best_model_path = OUTPUT_DIR / 'best_cnn.pth'

for epoch in range(N_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{N_EPOCHS} [Train]',
                              leave=False):
        imgs = imgs.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = sum(
            TASK_WEIGHTS[t] * criterion(outputs[t],
                                         torch.tensor(labels[t]).to(DEVICE))
            for t in TASKS
        )
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct_mod = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            loss = sum(
                TASK_WEIGHTS[t] * criterion(outputs[t],
                                             torch.tensor(labels[t]).to(DEVICE))
                for t in TASKS
            )
            val_loss += loss.item()
            preds = outputs['modulation'].argmax(dim=1).cpu()
            correct_mod += (preds == labels['modulation']).sum().item()
            total += len(labels['modulation'])

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    val_acc_mod = correct_mod / total * 100

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc_mod'].append(val_acc_mod)

    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | '
          f'Train Loss: {train_loss:.3f} | '
          f'Val Loss: {val_loss:.3f} | '
          f'Val Acc (Mod): {val_acc_mod:.1f}%')

    # Save best model
    if val_acc_mod > best_val_acc:
        best_val_acc = val_acc_mod
        torch.save(model.state_dict(), best_model_path)
        print(f'  ✓ Best model saved (acc={val_acc_mod:.1f}%)')

    scheduler.step()

print(f'\nTraining complete! Best Val Acc (Modulation): {best_val_acc:.1f}%')

## Cell 7 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss', color='#e74c3c')
ax1.plot(history['val_loss'],   label='Val Loss',   color='#3498db')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['val_acc_mod'], color='#2ecc71')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy (Modulation)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved')

## Cell 8 — Full Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_preds = {t: [] for t in TASKS}
all_true  = {t: [] for t in TASKS}
all_snr_db = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating'):
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        for t in TASKS:
            preds = outputs[t].argmax(dim=1).cpu().numpy()
            all_preds[t].extend(preds)
            all_true[t].extend(labels[t].numpy())
        all_snr_db.extend(labels['snr_db'].numpy())

# Accuracy per task
print('=== Test Accuracy per Task ===')
for t in TASKS:
    acc = np.mean(np.array(all_preds[t]) == np.array(all_true[t])) * 100
    print(f'  {t:15s}: {acc:.2f}%')

## Cell 9 — Confusion Matrix (Modulation)

In [ ]:
cm = confusion_matrix(all_true['modulation'], all_preds['modulation'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=MOD_CLASSES, yticklabels=MOD_CLASSES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix — Modulation Classification (Normalized)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_mod.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved')

## Cell 10 — Accuracy vs SNR

In [ ]:
# Bin test samples by SNR
snr_bins = np.arange(0, 32, 2)
bin_accs = []
bin_centers = []

snr_arr   = np.array(all_snr_db)
pred_arr  = np.array(all_preds['modulation'])
true_arr  = np.array(all_true['modulation'])

for i in range(len(snr_bins) - 1):
    lo, hi = snr_bins[i], snr_bins[i+1]
    mask = (snr_arr >= lo) & (snr_arr < hi)
    if mask.sum() > 0:
        acc = np.mean(pred_arr[mask] == true_arr[mask]) * 100
        bin_accs.append(acc)
        bin_centers.append((lo + hi) / 2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bin_centers, bin_accs, 'o-', color='#2ecc71', linewidth=2, markersize=6)
ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Modulation Classification Accuracy (%)', fontsize=12)
ax.set_title('CNN Accuracy vs SNR', fontsize=13)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'accuracy_vs_snr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Accuracy vs SNR saved')

## Cell 11 — Confusion Matrices (Impairment Tasks)

In [ ]:
task_classes = {
    'snr_range':    SNR_CLASSES,
    'phase_noise':  PN_CLASSES,
    'iq_imbalance': IQ_CLASSES,
    'jamming':      JAM_CLASSES,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (task, classes) in zip(axes.flat, task_classes.items()):
    cm = confusion_matrix(all_true[task], all_preds[task])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    acc = np.mean(np.array(all_preds[task]) == np.array(all_true[task])) * 100
    ax.set_title(f'{task} (acc={acc:.1f}%)', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices_tasks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Impairment confusion matrices saved')

## Cell 12 — Generalization Test (Unseen SNR Values)

In [ ]:
# Test whether the model generalizes to intermediate SNR values
# including values near the low/medium and medium/high SNR boundaries

snr_arr   = np.array(all_snr_db)
pred_arr  = np.array(all_preds['modulation'])
true_arr  = np.array(all_true['modulation'])

# Low/Medium/High boundary analysis
ranges = [
    ('Low SNR\n(0-5 dB)',    (0,  5)),
    ('Low-Med boundary\n(4-6 dB)',  (4,  6)),
    ('Medium SNR\n(5-15 dB)', (5, 15)),
    ('Med-High boundary\n(14-16 dB)', (14, 16)),
    ('High SNR\n(15-30 dB)', (15, 30)),
]

labels_r, accs_r = [], []
for label, (lo, hi) in ranges:
    mask = (snr_arr >= lo) & (snr_arr < hi)
    if mask.sum() > 0:
        acc = np.mean(pred_arr[mask] == true_arr[mask]) * 100
        labels_r.append(f'{label}\n(n={mask.sum()})')
        accs_r.append(acc)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels_r, accs_r, color=['#e74c3c','#f39c12','#2ecc71','#f39c12','#3498db'])
ax.set_ylabel('Modulation Accuracy (%)', fontsize=12)
ax.set_title('Generalization: Accuracy at SNR Boundaries', fontsize=13)
ax.set_ylim(0, 110)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5)
for bar, acc in zip(bars, accs_r):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'generalization_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Generalization test saved')

## Cell 13 — Save Results Summary

In [ ]:
results = {}
for t in TASKS:
    acc = np.mean(np.array(all_preds[t]) == np.array(all_true[t])) * 100
    results[t] = round(acc, 2)

results['best_val_acc_modulation'] = round(best_val_acc, 2)
results['total_params'] = total_params
results['n_epochs'] = N_EPOCHS

with open(OUTPUT_DIR / 'cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('=== Final CNN Results ===')
for k, v in results.items():
    print(f'  {k}: {v}')